# TabLint in Jupyter

**Spellcheck for tables, powered by TabPFN-3.5.** Inspect suspicious cells, review the model’s reasoning, and return a cleaned pandas DataFrame without leaving your notebook.

Choose **Run → Run All Cells** to start. This demo replays saved predictions on 398 real cars from UCI Auto MPG (CC BY 4.0). No model download, GPU or API key is needed. The eight injected typos and original values are documented in `demo/video/answer_key.json`.


In [ ]:
from pathlib import Path
import pandas as pd
import proofread
from proofread import Report

# Works from the example notebook or the launcher’s editable working copy.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "demo/video/auto_mpg_dirty.proofread.json").is_file())
report = Report.load(ROOT / "demo/video/auto_mpg_dirty.proofread.json")
df = report.data.copy()
df.head()


## 1. A value can look normal for its column and wrong for its row

The Honda Civic weighs **4,354 lb** in this table, despite having four cylinders, 53 hp and 33 mpg. TabPFN predicts each held-out cell from the other columns in its row. The saved report expects about **1,877 lb**, with an 80% plausible interval of **1,720–2,034 lb**. This is a model expectation; verify the source record before replacing a value.


In [ ]:
civic = report.issues[(report.issues.row == 181) & (report.issues.column == "weight")]
civic[["row", "column", "value", "suggested", "low", "high", "cause", "evidence"]]


## 2. Review directly in the table

Click a highlighted cell to inspect its expected value, uncertainty, likely cause and related columns. **Accept** applies a suggestion to the cleaned copy; **dismiss** leaves the source value unchanged; **undo** reverses the last decision in this widget session. You can toggle “only flagged rows.”

Try the Civic’s weight, then the Rabbit’s **0.0 horsepower** suggestion. Keyboard shortcuts after clicking the widget: `n` / `p` next / previous, `a` accept, `d` dismiss, `u` undo. Decisions sync back to Python.


In [ ]:
review = report.widget()
review.only_flagged = True
review.selected = int(civic.index[0])
review


## 3. Get your results back in pandas

After reviewing, run the next cell again to refresh the cleaned DataFrame and decision snapshot. The original `df` and saved report are unchanged. `review.decisions` includes the recorded value, suggested value, reason and current decision for every issue.


In [ ]:
cleaned = review.cleaned
decisions = review.decisions
display(decisions[["row", "column", "value", "suggested", "cause", "decision"]].head(10))
cleaned.loc[[175, 181], ["car", "horsepower", "weight", "origin"]]


### Export when you are ready

Run this snippet to save your cleaned table and review snapshot next to the working notebook. The snapshot records current decisions; notebook undo history is session-local.

```python
out = ROOT / "demo" / "notebook-workspace"
out.mkdir(parents=True, exist_ok=True)
review.cleaned.to_csv(out / "cleaned.csv", index=False)
review.decisions.to_csv(out / "decisions.csv", index=False)
```


## 4. Check your own data — numerical and categorical

The Auto MPG replay above includes numerical and origin-label issues. For a new DataFrame, TabLint also checks low-cardinality categorical columns with TabPFN’s classifier: it scores the probability of the recorded category given the rest of its held-out row, then offers a more likely alternative for review. Numeric cells use the regressor’s predictive distribution. Neither score is a calibrated probability that the source value is wrong.

This optional snippet runs **live inference**, requiring TabPFN weights and first-use model-license acceptance. CPU works; a GPU is faster. High-cardinality identifiers and free text are excluded from categorical checks.

```python
df = pd.read_csv("my_table.csv")
report = df.tablint.check(categorical=True)
review = report.widget()
review
```

Or use `df.tablint.view(categorical=True)` directly. Add `label="outcome"` if you also want to check a known target column. The original `df.proofread` accessor remains compatible.

[Getting started](https://github.com/James-Begin/TabLint/blob/main/docs/GETTING_STARTED.md) · [Evidence and limitations](https://github.com/James-Begin/TabLint/blob/main/docs/RESEARCH_OVERVIEW.md)
